# 🚦 Train CNN nhận diện biển báo GTSRB trên Google Colab

**Hướng dẫn:**
1. `Runtime → Change runtime type → GPU (T4)`.
2. Upload **mã nguồn** dự án (zip không cần data) ở mục 2.
3. Tải GTSRB dataset bằng 1 trong 3 cách ở mục 4 (Kaggle API / Drive / Upload zip).
4. Chạy lần lượt các cell còn lại.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Upload mã nguồn dự án

**Chỉ upload code (~vài MB), KHÔNG upload dataset.** Dataset GTSRB sẽ tải trực tiếp trên Colab ở mục 4.

### Cách đóng gói ở local (PowerShell trên Windows):
```powershell
# Chạy ở root D:\IT-HAU\KHMT\final_project
Compress-Archive `
  -Path src, app, scripts, notebooks, requirements.txt, `
        data\classes.txt, data\classes_en.txt, data\classes_vie.txt, data\README.md, `
        README.md, CHANGES.md `
  -DestinationPath final_project_code.zip -Force
```

### Hoặc trên macOS / Linux:
```bash
zip -r final_project_code.zip src app scripts notebooks \
    requirements.txt README.md CHANGES.md \
    data/classes.txt data/classes_en.txt data/classes_vie.txt data/README.md
```

**Cố ý loại bỏ** (sẽ tự sinh lại trên Colab):
- `GTSRB.zip` (611MB) → tải bằng Kaggle API ở mục 4A
- `data/processed/`, `data/gtsrb_raw/` → sinh ở mục 5
- `models/*.keras`, `reports/figures/` → sinh ở mục 6-7
- `__pycache__/`, `.ipynb_checkpoints/`, `.git/`, `.venv/`

→ File zip cuối cùng chỉ ~50-100KB, upload trong vài giây.

Sau đó chạy cell dưới để upload.

In [ ]:
from google.colab import files
import os, zipfile, shutil

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
extract_dir = '/content/project'
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(extract_dir)
subdirs = [d for d in os.listdir(extract_dir)
           if os.path.isdir(os.path.join(extract_dir, d))]
if len(subdirs) == 1 and not os.path.exists(os.path.join(extract_dir, 'src')):
    project_root = os.path.join(extract_dir, subdirs[0])
else:
    project_root = extract_dir
%cd $project_root
!ls -la

## 3. Cài dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Tải GTSRB dataset (chọn 1 trong 3 cách)

Sau khi chạy xong, file `GTSRB.zip` sẽ nằm ở root project (hoặc folder đã giải nén tại `data/gtsrb_raw/`).

### 4A. Kaggle API (khuyên dùng — nhanh nhất, ~1 phút)

Kaggle có **2 loại token** (UI mới 2024-2025). Dùng cách nào cũng được — Cách 1 sạch & an toàn hơn.

---

#### ✅ Cách 1: Colab Secrets + token mới (KHUYÊN DÙNG)

**Bước A — Lấy username & key từ Kaggle:**
1. Đăng nhập https://www.kaggle.com
2. Avatar góc phải → **`Settings`** (https://www.kaggle.com/settings)
3. Cuộn đến mục **`API`** → click **`Create New Token`**
4. Một **popup nhỏ** sẽ hiện ra với 2 thông tin (chính là cái bạn đang thấy):
   - **Username**: ví dụ `nguyenvana123`
   - **Key** (hoặc "API Token"): chuỗi dài kiểu `a1b2c3d4e5f6...`
5. Click **biểu tượng copy** bên cạnh từng dòng (Username & Key) — hoặc bôi đen rồi Ctrl+C.

**Bước B — Lưu vào Colab Secrets:**
1. Trong Colab, click **biểu tượng 🔑 (chìa khóa)** ở thanh dọc bên trái (gần icon Files).
2. Click **`+ Add new secret`** 2 lần để tạo 2 secret:
   - Name: `KAGGLE_USERNAME` | Value: dán username
   - Name: `KAGGLE_KEY` | Value: dán key
3. **Bật toggle `Notebook access`** (cột bên phải) cho cả 2 secret.

**Bước C — Vào trang dataset accept rules (chỉ làm 1 lần/đời):**
https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign → click **`I Understand and Accept`** nếu có.

Sau đó chạy cell ngay dưới đây.

In [ ]:
# === Cách 1: Colab Secrets (token mới) ===
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!pip install -q kaggle
!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign -p . --force
!mv -f gtsrb-german-traffic-sign.zip GTSRB.zip
!ls -lh GTSRB.zip

#### 🟡 Cách 2: Upload `kaggle.json` (token legacy)

Dùng nếu bạn không muốn xài Colab Secrets. UI mới đã ẩn nút download file đi — phải tìm đúng chỗ:

1. Vào https://www.kaggle.com/settings → cuộn xuống mục **`API`**.
2. **Cuộn tiếp xuống dưới mục `API`** sẽ thấy section **`Legacy API Credentials`** (chữ nhỏ hơn).
3. Click **`Create Legacy API Key`** — trình duyệt sẽ download file `kaggle.json` về máy.
   - Nếu không thấy section Legacy: click `Expire Token` ở mục API phía trên trước rồi thử lại.
4. Chạy cell dưới → khi hiện `Choose Files` → chọn `kaggle.json` từ Downloads.

> ⚠️ Không commit `kaggle.json` lên git.

In [ ]:
# === Cách 2: Upload kaggle.json (legacy) — bỏ comment nếu dùng ===
# from google.colab import files
# import os, shutil
#
# os.makedirs('/root/.kaggle', exist_ok=True)
# uploaded = files.upload()                # upload kaggle.json
# shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
# os.chmod('/root/.kaggle/kaggle.json', 0o600)
#
# !pip install -q kaggle
# !kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign -p . --force
# !mv -f gtsrb-german-traffic-sign.zip GTSRB.zip
# !ls -lh GTSRB.zip

### 4B. Mount Google Drive (nếu đã upload GTSRB.zip lên Drive)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/GTSRB.zip' ./GTSRB.zip
# !ls -lh GTSRB.zip

### 4C. Upload trực tiếp file GTSRB.zip (~611MB, chậm 10-15 phút)

In [ ]:
# from google.colab import files
# uploaded = files.upload()    # chọn GTSRB.zip
# !ls -lh GTSRB.zip

## 5. Tiền xử lý: giải nén + crop ROI + split train/val/test

Kết quả lưu ở `data/processed/{train,val,test}/<class_folder>/`.

In [ ]:
!python -m src.prepare_data

## 6. Huấn luyện CNN (IMG_SIZE=48, EPOCHS=60, class_weight cap=2.0)

In [ ]:
!python -m src.train

## 7. Đánh giá chi tiết

In [ ]:
!python -m src.evaluate

## 8. Hiển thị kết quả

In [ ]:
from IPython.display import Image, display
display(Image('reports/figures/training_curves.png'))
display(Image('reports/figures/confusion_matrix.png'))

## 9. Tải model + reports về máy

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/trained_model', 'zip', '.', 'models')
shutil.make_archive('/content/reports', 'zip', '.', 'reports')
files.download('/content/trained_model.zip')
files.download('/content/reports.zip')